# 03 - PCA (Principal Component Analysis)

**Author:** Beyza Şakrakdil  
**Based on:** AI in Chemical Engineering: Unlocking the Power Within Data (Romagnoli et al.)

In this notebook we learn PCA — one of the most used dimensionality reduction methods in process data analysis.


## 0. Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (9, 6)

print("Libraries ready.")


## 1. Create the data

We generate the same type of process data we used before.  
PCA is sensitive to scale, so we will standardize the data first.


In [ ]:
np.random.seed(42)
n = 300

df = pd.DataFrame({
    'temperature':   np.random.normal(85, 8, n),
    'pressure':      np.random.normal(2.5, 0.4, n),
    'flow_rate':     np.random.normal(120, 15, n),
    'concentration': np.random.normal(0.45, 0.08, n)
})

print("Shape:", df.shape)
df.head()


## 2. Why do we use PCA?

We have 4 variables. Sometimes we want to:
- Visualize the data in 2D
- Reduce noise
- Speed up later clustering or modeling

PCA finds new axes (principal components) that capture as much variance as possible.  
The first component (PC1) explains the most variance, PC2 the second most, and so on.


## 3. Scaling (very important!)

PCA works with distances / variances.  
If we don't scale, the variable with the largest numbers will dominate everything.
(If the temperature is around 100 and the pressure is around 2, for example, then the machine will think that temperature is more important)

We use StandardScaler.


In [ ]:
numeric_cols = ['temperature', 'pressure', 'flow_rate', 'concentration']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numeric_cols])

# Check that mean ≈ 0 and std ≈ 1
print("Mean of scaled data (should be close to 0):")
print(X_scaled.mean(axis=0).round(4))
print("\nStd of scaled data (should be close to 1):")
print(X_scaled.std(axis=0).round(4))


## 4. Apply PCA

We first fit PCA with 2 components so we can plot the result.


In [ ]:
# Create PCA object with 2 components
pca = PCA(n_components=2)

# Fit and transform in one step
X_pca = pca.fit_transform(X_scaled)

# Put the result into a DataFrame for easier plotting
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
print(pca_df.head())
print("\nShape after PCA:", pca_df.shape)


## 5. How much information did we keep?

`explained_variance_ratio_` tells us how much variance each component explains.


In [ ]:
print("Explained variance ratio (PC1, PC2):")
print(pca.explained_variance_ratio_.round(4))

print("\nTotal variance explained by 2 components:")
print(f"{pca.explained_variance_ratio_.sum():.2%}")


In [ ]:
# Bar plot of explained variance
plt.bar(['PC1', 'PC2'], pca.explained_variance_ratio_, color='steelblue', edgecolor='black')
plt.ylabel('Explained Variance Ratio')
plt.title('Variance Explained by Each Principal Component')
plt.ylim(0, 1)
plt.show()


## 6. Visualize the 2D PCA result


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(pca_df['PC1'], pca_df['PC2'], alpha=0.7, edgecolor='k', linewidth=0.3)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Data projected onto first two Principal Components')
plt.grid(True, alpha=0.3)
plt.show()


## 7. Loadings (what do the components mean?)

The **components_** attribute shows how much each original variable contributes to each PC.  
These are called loadings.


In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=numeric_cols
)

print("Loadings:")
print(loadings.round(3))


In [ ]:
# Heatmap of loadings
sns.heatmap(loadings, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('PCA Loadings')
plt.show()


## 8. Trying 3 components

Sometimes 2 components are not enough. Let's check with 3.


In [ ]:
pca3 = PCA(n_components=3)
X_pca3 = pca3.fit_transform(X_scaled)

print("Explained variance ratio (3 components):")
print(pca3.explained_variance_ratio_.round(4))
print(f"\nTotal variance explained: {pca3.explained_variance_ratio_.sum():.2%}")


In [ ]:
# Cumulative explained variance plot
pca_full = PCA().fit(X_scaled)   # all components

plt.plot(np.cumsum(pca_full.explained_variance_ratio_), marker='o')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('How many components do we need?')
plt.grid(True, alpha=0.3)
plt.axhline(0.9, color='red', linestyle='--', label='90% variance')
plt.legend()
plt.show()


## Summary

What we did:
- Standardized the process data (required for PCA)
- Applied PCA with 2 components and visualized the result
- Checked how much variance we kept
- Looked at the loadings to understand what each PC means
- Tested with 3 components and plotted cumulative variance

**Next:** t-SNE and UMAP (non-linear dimensionality reduction methods)
